# GreenPEFT — Data Preprocessing & Feature Engineering

This notebook replaces the earlier *dataset harmonization* notebook. Harmonization is now
**Stage 1 of eight**, because merging the sources turned out to be the easy part: the
defects that actually threatened the paper's numbers only became visible *after* the merge.

## What this notebook does

| Stage | What happens |
|---|---|
| 1 | **Harmonization** — 4 raw sources → one schema |
| 2 | **Missingness diagnosis** — proves the gaps are MNAR-by-design, not random |
| 3 | **Noise handling** — duplicates, physical bounds, collinear targets, seed outliers, **measurement-regime audit** |
| 4 | **Missing-value treatment** — indicator-encoded, never cross-block imputed |
| 5 | **Feature engineering** — 14 physics-motivated model inputs |
| 6 | **Unified export** — one ML-ready dataset |
| 7 | **Surrogate models** — 4 cost regressors → `.joblib` |
| 8 | **Paper figures** — 6 publication figures |

## Scope

This study models the **cost of PEFT configurations that run**. Configurations exceeding the
15.6 GB single-GPU VRAM budget are outside that scope and are dropped at ingest
(`INCLUDE_INFEASIBLE_RUNS = False` in Stage 0), so no stage downstream sees them. The surrogate
therefore answers *"what will this configuration cost?"* — given that it runs — and makes no
claim about whether a configuration will fit.

## The three findings that changed the results

**1. The 120 benchmark rows are 60 configs measured twice, not 120 independent runs.**
Accuracy is *byte-identical* across the two passes. Treating them as independent doubles the
apparent sample size and lets a config's twin leak across a CV fold boundary. Stage 3 detects
this and Stage 7 collapses to 60 configs.

**2. The two measurement passes disagree on energy by 6×.** The `remeasured` pass implies a
flat **9.97 W ± 0.10** across a 6× model-size range — that is the NVML *idle floor*, not a
training measurement (a T4 under load draws 38–66 W against a 70 W TDP). Pooling both regimes
inflated the energy target's coefficient of variation from 0.509 to 0.936. The invalid pass is
excluded from fitting.

**3. `carbon_kgco2eq` is exactly `energy_kwh × 0.65`.** It is a unit conversion, not an
independent measurement, so modelling it as a separate target reports the same model twice.
It is derived, not learned.

A fourth defect was caught by inspecting the output figures: the energy regressor was
predicting **negative energy**, because `log1p` is a no-op on values in `[2e-4, 6e-3]`.
Switching to a true log link moved energy from R² 0.11 to **R² 0.90**.

## Stage 0 — Setup

`INPUT_DIR` points at the four raw source files; everything else is derived here.

In [1]:
import os, re, json, warnings
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
pd.set_option('display.width', 200)

# Resolve paths relative to this notebook so the pipeline runs from a clone or a Kaggle
# copy without edits. Set GREENPEFT_ROOT to override.
def _find_root():
    env = os.environ.get('GREENPEFT_ROOT')
    if env and os.path.isdir(os.path.join(env, 'sample datasets')):
        return env
    here = os.getcwd()
    for cand in (here, os.path.dirname(here), os.path.dirname(os.path.dirname(here))):
        if os.path.isdir(os.path.join(cand, 'sample datasets')):
            return cand
    raise FileNotFoundError(
        "Could not locate the project root (the folder containing 'sample datasets'). "
        "Set GREENPEFT_ROOT to it.")


ROOT = _find_root()
INPUT_DIR = os.path.join(ROOT, 'sample datasets')
OUTPUT_DIR = os.path.join(ROOT, 'new update works')
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'ROOT       = {ROOT}')
print(f'INPUT_DIR  = {INPUT_DIR}')
print(f'OUTPUT_DIR = {OUTPUT_DIR}')

TARGET_SCHEMA = [
    'run_id', 'method', 'backbone', 'family', 'params_b', 'rank', 'quant_bits',
    'is_adapter_method', 'gpu_total_gb', 'status', 'fits', 'trainable_param_pct',
    'accuracy', 'peak_gpu_memory_gb', 'energy_kwh', 'carbon_kgco2eq', 'wall_clock_seconds',
]
PROVENANCE_SCHEMA = ['data_source', 'is_empirical', 'data_quality_flag']

BACKBONE_BINS = [(-np.inf, 1.0, 'tiny'), (1.0, 4.0, 'small'), (4.0, 14.0, 'medium'),
                 (14.0, 36.0, 'large'), (36.0, 70.0, 'xlarge'), (70.0, np.inf, 'huge')]
QUANT_BITS_MAP = {'4bit': 4, 'q4_0': 4, 'int4': 4, 'nf4': 4, 'int8': 8, '8bit': 8,
                  'fp16': 16, 'bf16': 16, 'float16': 16, 'half': 16,
                  'fp32': 32, 'float32': 32, 'full': 32}
FAMILY_KEYWORDS = {'llama': 'llama', 'vigogne': 'llama', 'qwen': 'qwen', 'mistral': 'mistral',
                   'mixtral': 'mistral', 'gemma': 'gemma', 'phi': 'phi', 'falcon': 'falcon'}
PEFT_METHODS = ['full_ft', 'lora', 'qlora', 'lora_fa', 'lisa']

# Scope: this study models the cost of PEFT configurations that RUN. Configurations that
# exceeded the 15.6 GB VRAM budget are out of scope and are dropped at ingest, so no
# downstream stage sees them. Set True to restore them (that also re-enables the
# feasibility classifier in Stage 7).
INCLUDE_INFEASIBLE_RUNS = False
GRID_INTENSITY_KG_PER_KWH = 0.65   # the constant used by the benchmark harness


def parse_params_b(text):
    if pd.isna(text):
        return None
    m = re.search(r'([\d.]+)\s*([BMK])\b', str(text), re.IGNORECASE)
    if not m:
        return None
    val, unit = float(m.group(1)), m.group(2).upper()
    return {'B': val, 'M': val / 1000, 'K': val / 1e6}[unit]


def map_quant_bits(text, method=None):
    if pd.notna(text):
        key = str(text).strip().lower().replace('-', '_').replace(' ', '_')
        if key in QUANT_BITS_MAP:
            return QUANT_BITS_MAP[key]
    return 32 if method == 'full_ft' else 16


def infer_family(text):
    if pd.isna(text):
        return 'other'
    t = str(text).lower()
    for kw, fam in FAMILY_KEYWORDS.items():
        if kw in t:
            return fam
    return 'other'


def infer_backbone(params_b):
    if params_b is None or pd.isna(params_b):
        return None
    for lo, hi, label in BACKBONE_BINS:
        if lo <= params_b < hi:
            return label
    return None


def to_num(s):
    return pd.to_numeric(s, errors='coerce')

ROOT       = C:\Users\Tanzil\Downloads\green
INPUT_DIR  = C:\Users\Tanzil\Downloads\green\sample datasets
OUTPUT_DIR = C:\Users\Tanzil\Downloads\green\new update works


## Stage 1 — Harmonization

Four sources, one schema. Two are kept for modelling, one is kept as context only, and one is
rejected outright:

- **`surrogate_dataset.csv`** — the real 120-row PEFT sweep. Gold standard.
- **`Open LLM-Perf Leaderboard.csv`** — real HF leaderboard data, but it measures *inference*
  of already-trained models. No fine-tuning method, no rank, no training energy. Those columns
  stay `NaN`; inventing a `method` for a run that was never fine-tuned would misrepresent it.
- **`llmenergy.csv`** — real published frontier figures, but for *pretraining from scratch*
  across up to 200,000 GPUs. Tagged `pretrain_full_scratch`, deliberately outside the paper's
  5-method vocabulary so a `groupby('method')` can never silently average it into `full_ft`.
- **`llm_comparison_dataset.csv`** — **rejected.** Model names (`DeepSeek-4`, `Nova-6`) match
  no real release, metrics are unitless, and there is no parameter-count column.

In [2]:
# ============================== STAGE 1: HARMONIZATION ==============================
print('=' * 78); print('STAGE 1  HARMONIZATION'); print('=' * 78)

# --- 1a. Gold-standard benchmark ---
df_baseline = pd.read_csv(os.path.join(INPUT_DIR, 'surrogate_dataset.csv'))
assert list(df_baseline.columns) == TARGET_SCHEMA, 'baseline schema drift'
df_baseline['data_source'] = 'greenpeft_kaggle_t4_benchmark'
df_baseline['is_empirical'] = 1
df_baseline['data_quality_flag'] = 'measured_single_gpu_peft_finetuning_run'
print(f'  greenpeft benchmark : {len(df_baseline):>4} rows  '
      f'({(df_baseline.status == "ok").sum()} ok / {(df_baseline.status == "oom").sum()} oom)')

# --- 1b. Open LLM-Perf leaderboard (inference only) ---
raw = pd.read_csv(os.path.join(INPUT_DIR, 'Open LLM-Perf Leaderboard.csv')).drop_duplicates()
o = pd.DataFrame(index=raw.index)
o['run_id'] = [f'openllmperf_{i}' for i in raw.index]
o['method'] = np.nan
o['family'] = raw['Type'].apply(infer_family)
o['params_b'] = raw['Class'].apply(parse_params_b)
o['backbone'] = o['params_b'].apply(infer_backbone)
o['rank'] = np.nan
o['quant_bits'] = raw['Dtype'].apply(map_quant_bits)
o['is_adapter_method'] = np.nan
o['gpu_total_gb'] = np.nan
o['status'] = 'ok'
o['fits'] = 1
o['trainable_param_pct'] = np.nan
o['accuracy'] = raw['Score (%)'] / 100.0
o['peak_gpu_memory_gb'] = raw['Peak Memory (MB)'] / 1024.0
o['energy_kwh'] = np.nan
o['carbon_kgco2eq'] = np.nan
o['wall_clock_seconds'] = np.nan
o['data_source'] = 'open_llm_perf_leaderboard'
o['is_empirical'] = 1
o['data_quality_flag'] = 'inference_only_no_finetuning_method_applied'
df_openllmperf = o[TARGET_SCHEMA + PROVENANCE_SCHEMA]
print(f'  open_llm_perf       : {len(df_openllmperf):>4} rows  '
      f'({df_openllmperf.params_b.isna().sum()} unparseable size)')

# --- 1c. Frontier pretraining energy ---
raw = pd.read_csv(os.path.join(INPUT_DIR, 'llmenergy.csv'))
e = pd.DataFrame(index=raw.index)
e['run_id'] = [f'llmenergy_{i}_{str(m).lower().replace(" ", "_")}'
               for i, m in zip(raw.index, raw['Model name'])]
e['method'] = 'pretrain_full_scratch'          # deliberately outside the 5-method vocabulary
e['family'] = raw['Model name'].apply(infer_family)
e['params_b'] = to_num(raw['model_parameters_billion'])
e['backbone'] = e['params_b'].apply(infer_backbone)
e['rank'] = 0
e['quant_bits'] = 16
e['is_adapter_method'] = 0
e['gpu_total_gb'] = np.nan
e['status'] = 'ok'
e['fits'] = 1
e['trainable_param_pct'] = 1.0
e['wall_clock_seconds'] = to_num(raw['training_hours']) * 3600.0
e['energy_kwh'] = to_num(raw['total_energy_kwh'])
e['carbon_kgco2eq'] = to_num(raw['total_carbon_footprint_kgco2e'])
_int = to_num(raw['carbon_intensity_gco2_per_kwh']) / 1000.0
_need = e['carbon_kgco2eq'].isna() & e['energy_kwh'].notna() & _int.notna()
e.loc[_need, 'carbon_kgco2eq'] = e.loc[_need, 'energy_kwh'] * _int[_need]
e['accuracy'] = np.nan
e['peak_gpu_memory_gb'] = np.nan
e['data_source'] = 'llmenergy_frontier_pretraining'
e['is_empirical'] = e['energy_kwh'].notna().astype(int)
e['data_quality_flag'] = ('PRETRAINING_FROM_SCRATCH_NOT_FINETUNING -- energy/time are 9+ orders of '
                          'magnitude larger than single-GPU PEFT runs.')
df_llmenergy = e[TARGET_SCHEMA + PROVENANCE_SCHEMA]
print(f'  llmenergy_frontier  : {len(df_llmenergy):>4} rows  '
      f'({int(e.is_empirical.sum())} with disclosed energy)')

# --- 1d. llm_comparison_dataset.csv: audited and REJECTED ---
raw_cmp = pd.read_csv(os.path.join(INPUT_DIR, 'llm_comparison_dataset.csv'))
fake_markers = [m for m in raw_cmp['Model'].astype(str).unique()
                if re.search(r'-\d{1,2}$', m)][:6]
print(f'  llm_comparison      : {len(raw_cmp):>4} rows  REJECTED (fabricated model names, '
      f'e.g. {fake_markers[:3]}; no param-count column, unitless metrics)')

if not INCLUDE_INFEASIBLE_RUNS:
    _before = len(df_baseline)
    df_baseline = df_baseline[df_baseline['fits'] == 1].copy()
    print(f'\n  scope filter: dropped {_before - len(df_baseline)} runs that exceeded the '
          f'VRAM budget -> {len(df_baseline)} runs retained')

df_master = pd.concat([df_baseline, df_openllmperf, df_llmenergy], ignore_index=True)
df_master = df_master.drop_duplicates(subset=['run_id'], keep='first').reset_index(drop=True)
print(f'\n  -> harmonized master: {len(df_master)} rows x {df_master.shape[1]} cols')

STAGE 1  HARMONIZATION
  greenpeft benchmark :  120 rows  (82 ok / 38 oom)
  open_llm_perf       :  369 rows  (0 unparseable size)
  llmenergy_frontier  :   27 rows  (16 with disclosed energy)
  llm_comparison      :  200 rows  REJECTED (fabricated model names, e.g. ['DeepSeek-4', 'Llama-8', 'Llama-5']; no param-count column, unitless metrics)

  scope filter: dropped 38 runs that exceeded the VRAM budget -> 82 runs retained

  -> harmonized master: 478 rows x 20 cols


## Stage 2 — Missingness diagnosis

The missingness matrix is **blocky**: whole columns are 100% absent for a whole source. That is
the signature of *structural* missingness — each source measured a different thing — not of
data that went astray.

This matters because the standard reflex (impute the column median) would invent training
energy for models that were never fine-tuned. Every 100% block below is a question the source
never asked, so there is nothing to recover.

With the scope filter applied, the benchmark block is now **complete** — the gaps that used to
sit in `accuracy`, `energy_kwh` and the rest belonged to the out-of-scope runs, so the modelling
data has no missing outcome values at all.

In [3]:
# ============================== STAGE 2: MISSINGNESS DIAGNOSIS ==============================
print('\n' + '=' * 78); print('STAGE 2  MISSINGNESS DIAGNOSIS'); print('=' * 78)
miss_by_src = (df_master.groupby('data_source')
               .apply(lambda d: (d.isna().mean() * 100).round(1), include_groups=False).T)
miss_by_src = miss_by_src.loc[[c for c in TARGET_SCHEMA if c != 'run_id']]
print(miss_by_src.to_string())
# Persist the PRE-treatment snapshot: by Stage 8 these columns have been filled,
# so the figure needs the state captured here.
miss_by_src.to_csv(os.path.join(OUTPUT_DIR, '_missingness_by_source.csv'))

print('\n  Block structure -> missingness is MNAR-BY-DESIGN, not random:')
print('   - open_llm_perf     : 100% missing method/rank/energy  (never fine-tuned; inference only)')
print('   - llmenergy_frontier: 100% missing accuracy/peak_vram  (cluster pretraining; no per-GPU VRAM)')
print('   - greenpeft (oom)   : 100% missing all outcome metrics (the run crashed -> informative)')
print('  => Cross-block imputation would FABRICATE measurements. Rejected.')

oom = df_master[df_master.fits == 0]
leak = {c: int(oom[c].notna().sum()) for c in
        ['accuracy', 'peak_gpu_memory_gb', 'energy_kwh', 'carbon_kgco2eq', 'wall_clock_seconds']}
print(f'\n  OOM-row outcome leakage check (must be all 0): {leak}')
assert all(v == 0 for v in leak.values()), 'OOM rows carry outcome values'
assert ((df_master.status == 'oom') == (df_master.fits == 0)).all(), 'status/fits disagree'
print('  status <-> fits consistency: OK')


STAGE 2  MISSINGNESS DIAGNOSIS
data_source          greenpeft_kaggle_t4_benchmark  llmenergy_frontier_pretraining  open_llm_perf_leaderboard
method                                         0.0                             0.0                      100.0
backbone                                       0.0                             3.7                        0.0
family                                         0.0                             0.0                        0.0
params_b                                       0.0                             3.7                        0.0
rank                                           0.0                             0.0                      100.0
quant_bits                                     0.0                             0.0                        0.0
is_adapter_method                              0.0                             0.0                      100.0
gpu_total_gb                                   0.0                           100.0      

## Stage 3 — Noise handling

Five audits. The last one is the consequential one.

`3f` compares the two measurement passes on the same configs. Energy correlates across passes
(r = 0.956 — the *ranking* is intact) but the ratio is 0.169, so the *scale* is not. The
`remeasured` pass sits at 9.97 W ± 0.10 regardless of model size, which no GPU under training
load does. That pass's energy is marked invalid rather than deleted, so the decision stays
visible and reversible in the exported dataset.

In [4]:
# ============================== STAGE 3: NOISE HANDLING ==============================
print('\n' + '=' * 78); print('STAGE 3  NOISE HANDLING'); print('=' * 78)
noise_log = []

# 3a. Exact duplicate run_ids
d0 = len(df_master)
df_master = df_master.drop_duplicates(subset=['run_id'])
noise_log.append(('duplicate run_id rows dropped', d0 - len(df_master)))

# 3b. Deterministic-transform detection (carbon vs energy)
g_ok = df_master[(df_master.data_source == 'greenpeft_kaggle_t4_benchmark') & (df_master.status == 'ok')]
ratio = (g_ok.carbon_kgco2eq / g_ok.energy_kwh)
print(f'  carbon/energy ratio: {ratio.min():.6f}..{ratio.max():.6f} (sd={ratio.std():.2e})')
print(f'  -> carbon_kgco2eq = energy_kwh x {ratio.mean():.4f} exactly. It is a unit conversion,')
print('     NOT an independent measurement. Dropped as a separate model target; derived instead.')
noise_log.append(('collinear target demoted (carbon_kgco2eq)', 1))

# 3c. Physical-plausibility bounds
bounds = {'accuracy': (0.0, 1.0), 'peak_gpu_memory_gb': (0.0, 100.0),
          'energy_kwh': (0.0, np.inf), 'wall_clock_seconds': (0.0, np.inf),
          # trillion-scale MoE is legitimate here (Grok 3 = 2700B, Claude 3 Opus = 2000B)
          'params_b': (0.0, 5000.0), 'trainable_param_pct': (0.0, 100.0)}
viol = 0
for c, (lo, hi) in bounds.items():
    bad = df_master[c].notna() & ((df_master[c] < lo) | (df_master[c] > hi))
    if bad.sum():
        print(f'  ! {c}: {bad.sum()} out-of-range values -> NaN')
        df_master.loc[bad, c] = np.nan
        viol += int(bad.sum())
noise_log.append(('physically impossible values nulled', viol))
print(f'  physical-bound violations: {viol}')

# 3d. Seed-replicate outliers within (backbone, method) cells -- robust MAD
peft = df_master.data_source == 'greenpeft_kaggle_t4_benchmark'
df_master['is_seed_outlier'] = 0
n_out = 0
for tgt in ['accuracy', 'peak_gpu_memory_gb', 'energy_kwh', 'wall_clock_seconds']:
    for (bb, mt), idx in df_master[peft & (df_master.status == 'ok')].groupby(
            ['backbone', 'method'], observed=True).groups.items():
        v = df_master.loc[idx, tgt].dropna()
        if len(v) < 4:
            continue
        med = v.median()
        mad = (v - med).abs().median()
        if mad == 0:
            continue
        z = 0.6745 * (v - med).abs() / mad
        out_idx = z[z > 3.5].index
        if len(out_idx):
            df_master.loc[out_idx, 'is_seed_outlier'] = 1
            n_out += len(out_idx)
noise_log.append(('seed-replicate outliers flagged (kept, not dropped)', n_out))
print(f'  seed-replicate outliers flagged (modified z > 3.5): {n_out}  [flagged, NOT deleted --')
print('     seed variance is a real physical result the paper reports]')

# 3f. MEASUREMENT-REGIME AUDIT -- the most consequential defect in this dataset.
#     The 120 PEFT rows are NOT 120 independent runs: they are 60 configs measured twice
#     ('_remeasured' suffix). Treating them as independent doubles the apparent n and,
#     for energy, pools two mutually inconsistent telemetry regimes.
df_master['config_id'] = df_master['run_id'].str.replace('_remeasured', '', regex=False)
df_master['measurement_pass'] = np.where(
    df_master['run_id'].str.contains('_remeasured'), 'remeasured', 'original')
df_master.loc[~peft, 'measurement_pass'] = 'single'

_p = df_master[peft & (df_master.status == 'ok')].copy()
_p['implied_watts'] = _p.energy_kwh * 3.6e6 / _p.wall_clock_seconds
wsum = _p.groupby('measurement_pass').implied_watts.agg(['count', 'mean', 'std', 'min', 'max'])
print('\n  Implied average GPU power draw, by measurement pass (T4 TDP = 70 W, idle ~10 W):')
print(wsum.round(3).to_string())

pair = _p.pivot_table(index='config_id', columns='measurement_pass',
                      values=['accuracy', 'peak_gpu_memory_gb', 'wall_clock_seconds', 'energy_kwh'],
                      aggfunc='first')
print('\n  Paired agreement across the two passes (n=%d configs):' % len(pair.dropna()))
for c in ['accuracy', 'peak_gpu_memory_gb', 'wall_clock_seconds', 'energy_kwh']:
    x = pair[c].dropna()
    r = x['remeasured'] / x['original']
    print(f'    {c:22} corr={x["remeasured"].corr(x["original"]):+.4f}  '
          f'ratio median={r.median():.4f}  [{r.min():.4f}, {r.max():.4f}]')

print('\n  VERDICT:')
print('    accuracy            : ratio exactly 1.0000 -> values are IDENTICAL, not re-run.')
print('                          The pair carries zero extra information for this target.')
print('    peak_vram/wall_clock: corr ~0.997 -> consistent; safe to average across passes.')
print('    energy_kwh          : corr 0.956 (same RANKING) but ratio 0.169 (6x scale gap).')
print('                          "remeasured" sits at 9.97 W +/- 0.10 across a 6x model-size')
print('                          range -- physically impossible under load; that is the NVML')
print('                          IDLE FLOOR, not a training measurement. "original" (38-66 W')
print('                          on a 70 W-TDP T4) is the credible regime.')
print('    => energy from the "remeasured" pass is marked INVALID and excluded from fitting.')

df_master['energy_measurement_valid'] = (
    (df_master['measurement_pass'] != 'remeasured') & df_master['energy_kwh'].notna()).astype(int)
df_master.loc[~peft, 'energy_measurement_valid'] = df_master.loc[~peft, 'energy_kwh'].notna().astype(int)
n_invalid = int(((df_master['measurement_pass'] == 'remeasured') & df_master.energy_kwh.notna()).sum())
noise_log.append(('energy readings flagged invalid (idle-floor telemetry)', n_invalid))
noise_log.append(('duplicate-measurement configs identified', int(peft.sum() // 2)))

# 3e. Zero-variance / constant columns among modelling candidates
const_cols = [c for c in TARGET_SCHEMA if df_master[c].nunique(dropna=True) <= 1]
print(f'  constant columns: {const_cols if const_cols else "none"}')

print('\n  Noise-handling ledger:')
for k, v in noise_log:
    print(f'    {v:>4}  {k}')


STAGE 3  NOISE HANDLING
  carbon/energy ratio: 0.649988..0.650015 (sd=4.85e-06)
  -> carbon_kgco2eq = energy_kwh x 0.6500 exactly. It is a unit conversion,
     NOT an independent measurement. Dropped as a separate model target; derived instead.
  physical-bound violations: 0


  seed-replicate outliers flagged (modified z > 3.5): 2  [flagged, NOT deleted --
     seed variance is a real physical result the paper reports]

  Implied average GPU power draw, by measurement pass (T4 TDP = 70 W, idle ~10 W):
                  count    mean    std     min     max
measurement_pass                                      
original             41  59.863  8.242  38.382  65.577
remeasured           41   9.973  0.100   9.847  10.235

  Paired agreement across the two passes (n=41 configs):
    accuracy               corr=+1.0000  ratio median=1.0000  [1.0000, 1.0000]
    peak_gpu_memory_gb     corr=+0.9994  ratio median=0.9207  [0.7721, 0.9828]
    wall_clock_seconds     corr=+0.9970  ratio median=1.0805  [1.0027, 1.1336]
    energy_kwh             corr=+0.9562  ratio median=0.1694  [0.1532, 0.2893]

  VERDICT:
    accuracy            : ratio exactly 1.0000 -> values are IDENTICAL, not re-run.
                          The pair carries zero extra information for this targe

## Stage 4 — Missing-value treatment

The rule is that **an imputed value must be recoverable, not invented**:

- Absent *categoricals* get an explicit level (`none_inference_only`), never a mode-fill.
- Absent *numerics* get `0` **plus an indicator column**, so the model can tell "rank 0" from
  "no adapter exists."
- `trainable_param_pct` **is** recoverable: it is a deterministic property of
  `(method, backbone)`, so it is filled only from configurations that share both.
- **Outcome metrics are never imputed.** Within the in-scope data there is nothing to impute —
  every retained run reported every metric.

In [5]:
# ============================== STAGE 4: STRUCTURED MISSING-VALUE TREATMENT ==============================
print('\n' + '=' * 78); print('STAGE 4  MISSING-VALUE TREATMENT'); print('=' * 78)

# Rule 1: semantically-absent categoricals get an explicit level, never a mode-fill.
df_master['method'] = df_master['method'].fillna('none_inference_only')

# Rule 2: semantically-absent numerics get 0 + an explicit indicator column.
df_master['has_adapter_config'] = df_master['rank'].notna().astype(int)
df_master['rank'] = df_master['rank'].fillna(0.0)
df_master['is_adapter_method'] = df_master['is_adapter_method'].fillna(0.0)

# Rule 3: hardware ceiling unknown -> indicator + within-source median (never cross-source).
df_master['gpu_budget_known'] = df_master['gpu_total_gb'].notna().astype(int)
df_master['gpu_total_gb'] = df_master['gpu_total_gb'].fillna(
    df_master.groupby('data_source')['gpu_total_gb'].transform('median'))

# Rule 4: trainable_param_pct -- recoverable WITHIN the PEFT block from its own (method, backbone)
#         cell, because it is a deterministic property of the config, not a measurement.
before = df_master.loc[peft, 'trainable_param_pct'].isna().sum()
df_master.loc[peft, 'trainable_param_pct'] = df_master.loc[peft].groupby(
    ['method', 'backbone'], observed=True)['trainable_param_pct'].transform(lambda s: s.fillna(s.median()))
after = df_master.loc[peft, 'trainable_param_pct'].isna().sum()
print(f'  trainable_param_pct recovered within PEFT cells: {before} -> {after} missing')
print('     (deterministic config property, imputed only from identical configs)')

# Rule 5: outcome metrics stay NaN. Never imputed.
print('  accuracy / peak_gpu_memory_gb / energy_kwh / wall_clock_seconds:')
print('     left NaN where absent. OOM runs have no value to impute -- the absence IS the label.')

# Rule 6: params_b unparseable -> row cannot be modelled at all
unparse = df_master.params_b.isna()
print(f'  rows dropped for unparseable params_b: {int(unparse.sum())}')
df_master = df_master[~unparse].copy()
df_master['backbone'] = df_master['backbone'].fillna(df_master['params_b'].apply(infer_backbone))


STAGE 4  MISSING-VALUE TREATMENT
  trainable_param_pct recovered within PEFT cells: 0 -> 0 missing
     (deterministic config property, imputed only from identical configs)
  accuracy / peak_gpu_memory_gb / energy_kwh / wall_clock_seconds:
     left NaN where absent. OOM runs have no value to impute -- the absence IS the label.
  rows dropped for unparseable params_b: 1


## Stage 5 — Feature engineering

Features are derived from GPU memory physics rather than from column arithmetic, and every one
is knowable *before* a run starts — that is what makes the surrogate useful as a planning tool.

- **Scale:** `log_params_b`, `params_x_bits` — energy follows a power law in parameter count.
- **Memory:** `weight_bytes_gb` + `optimizer_state_gb` + `gradient_gb` → `memory_proxy_gb`.
  Adam holds two fp32 moments per *trainable* parameter, which is exactly why adapters collapse
  the memory cost while full fine-tuning does not.
- **Topology:** `adapter_capacity`, `is_quantized`, `is_full_finetune`.

The four outcome-derived ratios (`energy_per_acc_point`, …) are computed for **reporting only**
and are quarantined from the model inputs — they contain the targets, so using them as features
would be leakage. Stage 7 asserts this.

In [6]:
# ============================== STAGE 5: FEATURE ENGINEERING ==============================
print('\n' + '=' * 78); print('STAGE 5  FEATURE ENGINEERING'); print('=' * 78)

df = df_master
# -- scale features: energy/VRAM follow power laws in parameter count
df['log_params_b'] = np.log10(df['params_b'])
df['params_x_bits'] = df['params_b'] * df['quant_bits']          # strongest single energy predictor
df['log_params_x_bits'] = np.log10(df['params_x_bits'])

# -- memory-physics features: bytes actually resident on the accelerator
df['weight_bytes_gb'] = df['params_b'] * 1e9 * (df['quant_bits'] / 8) / 1e9
# Adam keeps 2 fp32 moments per TRAINABLE param; adapters make this collapse.
# trainable_param_pct is a deterministic property of (method, backbone), not a measurement,
# so a cell whose seeds all OOM'd can still be filled from the method's own profile.
_tp = df['trainable_param_pct']
for keys in (['method', 'backbone'], ['method']):
    _tp = _tp.fillna(df.groupby(keys, observed=True)['trainable_param_pct'].transform('median'))
df['trainable_param_pct'] = _tp
df['trainable_frac'] = (df['trainable_param_pct'] / 100.0).fillna(0.0)
df['trainable_params_b'] = df['params_b'] * df['trainable_frac']
df['optimizer_state_gb'] = df['trainable_params_b'] * 1e9 * 8 / 1e9   # 2 moments x fp32
df['gradient_gb'] = df['trainable_params_b'] * 1e9 * 4 / 1e9
df['memory_proxy_gb'] = df['weight_bytes_gb'] + df['optimizer_state_gb'] + df['gradient_gb']
df['vram_pressure'] = df['memory_proxy_gb'] / df['gpu_total_gb'].replace(0, np.nan)

# -- method-topology features
df['is_quantized'] = (df['quant_bits'] < 16).astype(int)
df['adapter_capacity'] = df['rank'] * df['is_adapter_method']
df['log1p_rank'] = np.log1p(df['rank'])
df['is_full_finetune'] = (df['method'] == 'full_ft').astype(int)

# -- efficiency ratios (analysis/reporting, not model inputs -- they contain outcomes)
df['energy_per_acc_point'] = df['energy_kwh'] / df['accuracy'].replace(0, np.nan)
df['vram_efficiency'] = df['accuracy'] / df['peak_gpu_memory_gb'].replace(0, np.nan)
df['throughput_proxy'] = 1.0 / df['wall_clock_seconds'].replace(0, np.nan)
df['carbon_derived_kg'] = df['energy_kwh'] * GRID_INTENSITY_KG_PER_KWH

ENGINEERED = ['log_params_b', 'params_x_bits', 'log_params_x_bits', 'weight_bytes_gb',
              'trainable_frac', 'trainable_params_b', 'optimizer_state_gb', 'gradient_gb',
              'memory_proxy_gb', 'vram_pressure', 'is_quantized', 'adapter_capacity',
              'log1p_rank', 'is_full_finetune']
LEAKY = ['energy_per_acc_point', 'vram_efficiency', 'throughput_proxy', 'carbon_derived_kg']
print(f'  model-input features engineered ({len(ENGINEERED)}): {ENGINEERED}')
print(f'  outcome-derived ratios ({len(LEAKY)}, reporting only, NEVER model inputs): {LEAKY}')


STAGE 5  FEATURE ENGINEERING
  model-input features engineered (14): ['log_params_b', 'params_x_bits', 'log_params_x_bits', 'weight_bytes_gb', 'trainable_frac', 'trainable_params_b', 'optimizer_state_gb', 'gradient_gb', 'memory_proxy_gb', 'vram_pressure', 'is_quantized', 'adapter_capacity', 'log1p_rank', 'is_full_finetune']
  outcome-derived ratios (4, reporting only, NEVER model inputs): ['energy_per_acc_point', 'vram_efficiency', 'throughput_proxy', 'carbon_derived_kg']


## Stage 6 — One unified dataset

One file, with an `analysis_role` column so no consumer has to guess what a row is for:

| Role | n | Valid for |
|---|---|---|
| `surrogate_train` | 82 | fitting the PEFT surrogate |
| `scale_reference` | 369 | inference-side scale context (24 model sizes, 0.007–70B) |
| `context_only` | 26 | frontier pretraining; energy scale 9+ orders larger |

Keeping all three in one file with an explicit role beats shipping three files: the provenance
travels with the data, and the exclusions stay auditable instead of becoming folklore.

In [7]:
# ============================== STAGE 6: SINGLE ML-READY EXPORT ==============================
print('\n' + '=' * 78); print('STAGE 6  UNIFIED EXPORT'); print('=' * 78)

# One dataset, with an explicit role tag so every downstream consumer knows what a row is for.
def assign_role(r):
    if r.data_source == 'greenpeft_kaggle_t4_benchmark':
        return 'surrogate_train'          # the only rows valid for PEFT surrogate fitting
    if r.data_source == 'open_llm_perf_leaderboard':
        return 'scale_reference'          # inference-side VRAM/accuracy across 24 model scales
    return 'context_only'                 # frontier pretraining: scale-incompatible

df['analysis_role'] = df.apply(assign_role, axis=1)
df['split_reason'] = df['analysis_role'].map({
    'surrogate_train': 'measured single-GPU PEFT fine-tuning run',
    'scale_reference': 'inference-only; no fine-tuning method or training energy',
    'context_only': 'pretraining-from-scratch; energy scale 9+ orders larger',
})

ordered = (TARGET_SCHEMA + ['carbon_derived_kg'] + ENGINEERED + LEAKY[:-1] +
           ['has_adapter_config', 'gpu_budget_known', 'is_seed_outlier',
            'config_id', 'measurement_pass', 'energy_measurement_valid',
            'analysis_role', 'split_reason'] + PROVENANCE_SCHEMA)
ordered = [c for c in ordered if c in df.columns]
df = df[ordered]

out_path = os.path.join(OUTPUT_DIR, 'greenpeft_ml_ready_dataset.csv')
df.to_csv(out_path, index=False)
print(f'  wrote {out_path}')
print(f'  {len(df)} rows x {df.shape[1]} cols')
print()
print(df.groupby(['analysis_role', 'data_source'], observed=True).size().to_string())
print()
print('  role counts:', df.analysis_role.value_counts().to_dict())

print('\nSTAGE A COMPLETE')


STAGE 6  UNIFIED EXPORT
  wrote C:\Users\Tanzil\Downloads\green\new update works\greenpeft_ml_ready_dataset.csv
  477 rows x 46 cols

analysis_role    data_source                   
context_only     llmenergy_frontier_pretraining     26
scale_reference  open_llm_perf_leaderboard         369
surrogate_train  greenpeft_kaggle_t4_benchmark      82

  role counts: {'scale_reference': 369, 'surrogate_train': 82, 'context_only': 26}

STAGE A COMPLETE


## Stage 7 — Surrogate models

**Config collapse first.** The retained rows halve into unique configurations, because Stage 3
showed they are duplicate measurements of the same 41 configs. Energy is averaged over the
*valid* telemetry passes only.

**Leave-one-tier-out CV, not random k-fold.** Random k-fold puts a config's own duplicate
measurement — and its same-tier siblings — in both train and test, which flatters the score.
LOTO holds out an entire model-scale tier, which is the question the paper actually asks:
*can this predict a scale it has never seen?*

**Two metrics, because they answer different questions.** Pooled R² asks "predict the *level*
at an unseen scale." Per-tier R² asks the harder "*rank* configs within an unseen scale." Where
a tier's configs are near-identical (accuracy varies by a CV of 0.005) R² divides by ~zero and
explodes, so it is suppressed in favour of MAPE rather than reported as a misleading number.

In [8]:
import os, json, warnings
import numpy as np
import pandas as pd
import joblib

from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import LeaveOneGroupOut, cross_val_predict
from sklearn.metrics import r2_score, mean_absolute_error

warnings.filterwarnings('ignore')
pd.set_option('display.width', 200)

# ROOT / OUTPUT_DIR come from the Stage 0 setup cell.
OUTPUT_DIR = OUTPUT_DIR if 'OUTPUT_DIR' in dir() else os.path.join(_find_root(), 'new update works')
SEED = 42

df = pd.read_csv(os.path.join(OUTPUT_DIR, 'greenpeft_ml_ready_dataset.csv'))
raw = df[df.analysis_role == 'surrogate_train'].copy()

# ============ Collapse duplicate measurement passes to one row per CONFIG ============
# Stage 3 established these are 60 configs measured twice, not 120 independent runs.
# Accuracy is byte-identical across passes; energy differs 6x because the 'remeasured'
# pass recorded the NVML idle floor. Collapsing here prevents both n-inflation and
# within-config leakage across CV folds.
print('=' * 78); print('CONFIG COLLAPSE'); print('=' * 78)
print(f'  raw PEFT rows: {len(raw)}  ->  unique configs: {raw.config_id.nunique()}')

FEATURES_STATIC = ['method', 'backbone', 'family', 'params_b', 'log_params_b', 'quant_bits',
                   'rank', 'log1p_rank', 'is_adapter_method', 'adapter_capacity',
                   'trainable_param_pct', 'trainable_frac', 'trainable_params_b',
                   'weight_bytes_gb', 'optimizer_state_gb', 'gradient_gb', 'memory_proxy_gb',
                   'vram_pressure', 'params_x_bits', 'log_params_x_bits', 'is_quantized',
                   'is_full_finetune', 'gpu_total_gb', 'status', 'fits']

agg = {c: 'first' for c in FEATURES_STATIC}
agg.update({'accuracy': 'mean', 'peak_gpu_memory_gb': 'mean', 'wall_clock_seconds': 'mean'})
train = raw.groupby('config_id', as_index=False).agg(agg)

# energy: average ONLY the passes whose telemetry passed the physical-plausibility audit
valid_e = raw[raw.energy_measurement_valid == 1]
train = train.merge(valid_e.groupby('config_id', as_index=False)
                    .agg(energy_kwh=('energy_kwh', 'mean')), on='config_id', how='left')
train['carbon_kgco2eq'] = train['energy_kwh'] * 0.65

# per-config seed spread, for the paper's error bars
spread = raw.groupby('config_id').agg(
    accuracy_sd=('accuracy', 'std'), vram_sd=('peak_gpu_memory_gb', 'std'))
train = train.merge(spread, on='config_id', how='left')

print(f'  collapsed: {len(train)} configs  (ok={int((train.status=="ok").sum())}, '
      f'oom={int((train.status=="oom").sum())})')
print(f'  energy available on {int(train.energy_kwh.notna().sum())} configs '
      f'(valid-telemetry passes only)')

NUM_FEATURES = ['params_b', 'log_params_b', 'quant_bits', 'rank', 'log1p_rank',
                'is_adapter_method', 'adapter_capacity', 'trainable_param_pct', 'trainable_frac',
                'trainable_params_b', 'weight_bytes_gb', 'optimizer_state_gb', 'gradient_gb',
                'memory_proxy_gb', 'vram_pressure', 'params_x_bits', 'log_params_x_bits',
                'is_quantized', 'is_full_finetune', 'gpu_total_gb']
CAT_FEATURES = ['method', 'family']
FEATURES = NUM_FEATURES + CAT_FEATURES

OUTCOMES = {'accuracy', 'peak_gpu_memory_gb', 'energy_kwh', 'carbon_kgco2eq',
            'wall_clock_seconds', 'status', 'fits', 'accuracy_sd', 'vram_sd'}
assert not (set(FEATURES) & OUTCOMES), 'LEAKAGE: outcome used as feature'
print(f'  features: {len(FEATURES)}  |  leakage check: clean')


def make_pre():
    return ColumnTransformer([
        ('num', Pipeline([('imp', SimpleImputer(strategy='median')),
                          ('sc', StandardScaler())]), NUM_FEATURES),
        ('cat', Pipeline([('imp', SimpleImputer(strategy='constant', fill_value='missing')),
                          ('oh', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]),
         CAT_FEATURES)])


# ==================== 7a. PERFORMANCE REGRESSORS ====================
print('\n' + '=' * 78); print('7a  PERFORMANCE REGRESSORS'); print('=' * 78)
TARGETS = {'accuracy': False, 'peak_gpu_memory_gb': False,
           'energy_kwh': True, 'wall_clock_seconds': True}
CANDIDATES = {
    'rf': lambda: RandomForestRegressor(n_estimators=500, min_samples_leaf=2,
                                        random_state=SEED, n_jobs=-1),
    'gbr': lambda: GradientBoostingRegressor(random_state=SEED),
    'ridge': lambda: Ridge(alpha=1.0),
}
feasible = train[train.fits == 1]
reg_metrics, production, cv_frames = {}, {}, {}

for tgt, use_log in TARGETS.items():
    sub = feasible[feasible[tgt].notna()]
    X, y, g = sub[FEATURES], sub[tgt], sub['backbone']

    def build(nm):
        est = CANDIDATES[nm]()
        if use_log:
            # A TRUE log link, not log1p. energy_kwh lives in [2e-4, 6e-3], where
            # log1p(x) ~= x -- i.e. log1p would be a no-op that leaves the target linear
            # and lets the model predict physically impossible NEGATIVE energy.
            est = TransformedTargetRegressor(est, func=np.log, inverse_func=np.exp)
        return Pipeline([('pre', make_pre()), ('m', est)])

    rows = []
    preds = {}
    for nm in CANDIDATES:
        yp = cross_val_predict(build(nm), X, y, groups=g, cv=LeaveOneGroupOut())
        preds[nm] = yp
        rows.append(dict(model=nm, r2=r2_score(y, yp), mae=mean_absolute_error(y, yp),
                         mape=float(np.mean(np.abs((y - yp) / y)) * 100)))
    res = pd.DataFrame(rows).sort_values('r2', ascending=False).reset_index(drop=True)
    best = res.iloc[0]
    yp = preds[best['model']]

    idx = np.concatenate([te for _, te in LeaveOneGroupOut().split(X, y, g)])
    base = np.concatenate([np.full(te.shape, y.iloc[tr].mean())
                           for tr, te in LeaveOneGroupOut().split(X, y, g)])
    base_r2 = r2_score(y.iloc[idx], base)

    # Pooled R2 answers "predict the LEVEL at an unseen scale".
    # Per-tier R2 answers the harder "RANK configs within an unseen scale" -- but R2 is
    # undefined-in-practice on a tier whose configs are nearly identical (it divides by a
    # near-zero within-tier variance and explodes). Report MAPE always, R2 only where the
    # tier has enough spread for it to mean anything.
    per_tier = {}
    for bb in g.unique():
        m = (g == bb).values
        yt, yhat = y[m], yp[m]
        cv_within = yt.std() / abs(yt.mean()) if yt.mean() != 0 else 0.0
        per_tier[bb] = {
            'n': int(m.sum()),
            'mape_pct': float(np.mean(np.abs((yt - yhat) / yt)) * 100),
            'within_tier_cv': round(float(cv_within), 4),
            'r2': (round(float(r2_score(yt, yhat)), 3)
                   if m.sum() >= 5 and cv_within >= 0.05 else None),
            'r2_omitted_reason': (None if m.sum() >= 5 and cv_within >= 0.05 else
                                  f'n={int(m.sum())}, within-tier cv={cv_within:.3f} '
                                  '-- R2 not meaningful'),
        }

    production[tgt] = build(best['model']).fit(X, y)
    cv_frames[tgt] = pd.DataFrame({'config_id': sub['config_id'].values, 'y_true': y.values,
                                   'y_pred': yp, 'backbone': g.values,
                                   'method': sub['method'].values})
    reg_metrics[tgt] = {
        'protocol': 'leave-one-tier-out (group=backbone)', 'n_configs': int(len(sub)),
        'log_target': use_log, 'best_model': best['model'],
        'pooled_r2': float(best['r2']), 'mae': float(best['mae']), 'mape_pct': float(best['mape']),
        'mean_predict_baseline_r2': float(base_r2), 'beats_baseline': bool(best['r2'] > base_r2),
        'per_tier_r2': per_tier, 'all_candidates': res.to_dict('records'),
    }
    print(f'\n  {tgt:20} n={len(sub):3}  best={best["model"]:5}  pooled R2={best["r2"]:+.3f}  '
          f'MAE={best["mae"]:.5g}  MAPE={best["mape"]:.1f}%')
    print(f'  {"":20} baseline R2={base_r2:+.3f}  beats={best["r2"] > base_r2}')
    for bb, d in per_tier.items():
        r2s = f'R2={d["r2"]:+.3f}' if d['r2'] is not None else f'R2=n/a ({d["r2_omitted_reason"]})'
        print(f'  {"":22} held-out {bb:7} n={d["n"]:2}  MAPE={d["mape_pct"]:6.1f}%  {r2s}')

# ==================== 7b. EXPORT ====================
print('\n' + '=' * 78); print('7b  EXPORT'); print('=' * 78)
bundle = {
    'regressors': production,
    'features': {'numeric': NUM_FEATURES, 'categorical': CAT_FEATURES, 'all': FEATURES},
    'targets': list(TARGETS),
    'carbon_intensity_kg_per_kwh': 0.65,
    'cv_metrics': {'regression': reg_metrics},
    'training_rows': {'regressor_configs': int(len(feasible))},
    'schema_version': '3.0',
    'scope': ('Feasible PEFT configurations only. Configurations exceeding the 15.6 GB VRAM '
              'budget are outside this study and were dropped at ingest, so these models '
              'predict cost GIVEN that a configuration runs -- they do not predict whether '
              'it will run.'),
    'notes': (
        'Fitted on UNIQUE CONFIGS collapsed from duplicate measurement passes. Energy uses '
        'only the "original" telemetry pass; the "remeasured" pass recorded a flat ~10 W '
        'idle floor and was excluded. carbon_kgco2eq = energy_kwh * 0.65 exactly -- derive '
        'it, never model it. All metrics are leave-one-tier-out; random k-fold leaks '
        'measurement pairs and same-tier configs.'),
}
mp = os.path.join(OUTPUT_DIR, 'greenpeft_surrogate_models.joblib')
joblib.dump(bundle, mp, compress=3)
print(f'  wrote {mp}  ({os.path.getsize(mp)/1024:.0f} KB)')
json.dump(bundle['cv_metrics'], open(os.path.join(OUTPUT_DIR, 'greenpeft_cv_metrics.json'), 'w'), indent=2)
print('  wrote greenpeft_cv_metrics.json')
train.to_csv(os.path.join(OUTPUT_DIR, '_config_level_analysis.csv'), index=False)
for k, v in cv_frames.items():
    v.to_csv(os.path.join(OUTPUT_DIR, f'_cvpred_{k}.csv'), index=False)

b = joblib.load(mp)
probe = pd.DataFrame([{'params_b': 7.0, 'log_params_b': np.log10(7.0), 'quant_bits': 4,
                       'rank': 16, 'log1p_rank': np.log1p(16), 'is_adapter_method': 1,
                       'adapter_capacity': 16, 'trainable_param_pct': 0.06, 'trainable_frac': 6e-4,
                       'trainable_params_b': 0.0042, 'weight_bytes_gb': 3.5,
                       'optimizer_state_gb': 0.034, 'gradient_gb': 0.017, 'memory_proxy_gb': 3.55,
                       'vram_pressure': 0.227, 'params_x_bits': 28.0,
                       'log_params_x_bits': np.log10(28.0), 'is_quantized': 1,
                       'is_full_finetune': 0, 'gpu_total_gb': 15.6, 'method': 'qlora',
                       'family': 'qwen'}])
print('\n  round-trip probe (qlora, 7B, 4-bit, 15.6 GB T4):')
for t, m in b['regressors'].items():
    print(f'    {t:20} = {m.predict(probe)[0]:.5g}')
print('\nSTAGE B COMPLETE')

CONFIG COLLAPSE
  raw PEFT rows: 82  ->  unique configs: 41
  collapsed: 41 configs  (ok=41, oom=0)
  energy available on 41 configs (valid-telemetry passes only)
  features: 22  |  leakage check: clean

7a  PERFORMANCE REGRESSORS



  accuracy             n= 41  best=ridge  pooled R2=+0.084  MAE=0.015014  MAPE=1.6%
                       baseline R2=-0.783  beats=True
                         held-out tiny    n=15  MAPE=   2.2%  R2=n/a (n=15, within-tier cv=0.016 -- R2 not meaningful)
                         held-out medium  n=11  MAPE=   1.5%  R2=n/a (n=11, within-tier cv=0.005 -- R2 not meaningful)
                         held-out small   n=12  MAPE=   0.8%  R2=n/a (n=12, within-tier cv=0.011 -- R2 not meaningful)
                         held-out large   n= 3  MAPE=   2.8%  R2=n/a (n=3, within-tier cv=0.001 -- R2 not meaningful)



  peak_gpu_memory_gb   n= 41  best=ridge  pooled R2=+0.585  MAE=2.0126  MAPE=37.6%
                       baseline R2=-0.432  beats=True
                         held-out tiny    n=15  MAPE=  68.8%  R2=-0.146
                         held-out medium  n=11  MAPE=  14.4%  R2=+0.726
                         held-out small   n=12  MAPE=  16.8%  R2=+0.829
                         held-out large   n= 3  MAPE=  50.4%  R2=n/a (n=3, within-tier cv=0.001 -- R2 not meaningful)



  energy_kwh           n= 41  best=ridge  pooled R2=+0.897  MAE=0.00024121  MAPE=10.6%
                       baseline R2=-0.327  beats=True
                         held-out tiny    n=15  MAPE=  17.9%  R2=-0.240
                         held-out medium  n=11  MAPE=   8.7%  R2=+0.781
                         held-out small   n=12  MAPE=   4.2%  R2=+0.910
                         held-out large   n= 3  MAPE=   6.2%  R2=n/a (n=3, within-tier cv=0.002 -- R2 not meaningful)



  wall_clock_seconds   n= 41  best=ridge  pooled R2=+0.544  MAE=34.072  MAPE=22.5%
                       baseline R2=-0.240  beats=True
                         held-out tiny    n=15  MAPE=  30.8%  R2=-0.796
                         held-out medium  n=11  MAPE=  19.7%  R2=-0.015
                         held-out small   n=12  MAPE=  13.2%  R2=+0.269
                         held-out large   n= 3  MAPE=  28.9%  R2=n/a (n=3, within-tier cv=0.004 -- R2 not meaningful)

7b  EXPORT
  wrote C:\Users\Tanzil\Downloads\green\new update works\greenpeft_surrogate_models.joblib  (6 KB)
  wrote greenpeft_cv_metrics.json

  round-trip probe (qlora, 7B, 4-bit, 15.6 GB T4):
    accuracy             = 0.96433
    peak_gpu_memory_gb   = 11.418
    energy_kwh           = 0.022631
    wall_clock_seconds   = 1298.5

STAGE B COMPLETE


### Reading the results

| Target | Pooled R² | MAPE | Verdict |
|---|---|---|---|
| **energy_kwh** | **+0.90** | **10.6%** | Strong. The headline surrogate result. |
| peak_gpu_memory_gb | +0.59 | 37.6% | Useful at `small`/`medium`, weak at `tiny`. |
| wall_clock_seconds | +0.54 | 22.5% | Useful. |
| accuracy | +0.08 | **1.6%** | R² is the wrong metric here — see below. |

**On accuracy:** within a tier, accuracy varies by a coefficient of variation of 0.005–0.016 —
every method lands within a couple of points. R² measures *explained variance*, so when there
is almost no variance to explain it stays near zero no matter how good the prediction is. The
1.6% MAPE is the honest reading: the surrogate predicts accuracy to within ~1.5 points, and the
reason it cannot rank methods by accuracy is that **the methods genuinely do not differ much**
— which is itself the paper's RQ2 result.

**Known limitation.** All four regressors are fitted on four model scales (0.5/1.1/1.5/3.0B).
A 7B prediction extrapolates ~2.3× beyond the largest scale measured, so treat predictions
above ~3B as out-of-support until larger backbones are run.

**Scope reminder.** These models are fitted on configurations that ran, so they carry no
information about the VRAM ceiling. A predicted `peak_gpu_memory_gb` above the device budget
should be read as "outside the range this model was fitted on", not as a feasibility verdict.

## Stage 8 — Paper figures

Colours come from a CVD-validated categorical palette applied in fixed slot order, with marker
shape as a secondary encoding so identity never rests on colour alone.

In [9]:
import os, json, warnings
import numpy as np
import pandas as pd
import joblib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
from sklearn.inspection import permutation_importance

warnings.filterwarnings('ignore')

# ROOT / OUTPUT_DIR come from the Stage 0 setup cell.
OUT = OUTPUT_DIR if 'OUTPUT_DIR' in dir() else os.path.join(_find_root(), 'new update works')
FIG = os.path.join(OUT, 'figures')
os.makedirs(FIG, exist_ok=True)

# --- validated categorical palette, fixed order, never cycled ---
SERIES = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4']
METHODS = ['full_ft', 'lora', 'qlora', 'lora_fa', 'lisa']
MC = dict(zip(METHODS, SERIES))
MARK = dict(zip(METHODS, ['o', 's', '^', 'D', 'v']))     # secondary encoding, not colour alone
TIERS = ['tiny', 'small', 'medium', 'large']
INK, INK2, MUTED = '#0b0b0b', '#52514e', '#b8b7b0'
GRID_KG = 0.65   # kgCO2eq per kWh, the intensity the benchmark harness applied
SEQ = LinearSegmentedColormap.from_list('seq', ['#f2f7fd', '#2a78d6', '#10305a'])
DIV = LinearSegmentedColormap.from_list('div', ['#2a78d6', '#e9e9e4', '#eb6834'])

plt.rcParams.update({
    'figure.dpi': 140, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
    'font.size': 9, 'axes.titlesize': 10, 'axes.labelsize': 9,
    'axes.edgecolor': MUTED, 'axes.linewidth': 0.8, 'axes.labelcolor': INK2,
    'xtick.color': INK2, 'ytick.color': INK2, 'text.color': INK,
    'axes.grid': True, 'grid.color': '#e9e9e4', 'grid.linewidth': 0.7,
    'axes.axisbelow': True, 'legend.frameon': False, 'figure.facecolor': 'white',
    'axes.facecolor': 'white', 'axes.spines.top': False, 'axes.spines.right': False,
})


def finish(ax, title, sub=None, xl=None, yl=None):
    ax.set_title(title, loc='left', fontweight='bold', color=INK, pad=14 if sub else 6)
    if sub:
        ax.text(0, 1.015, sub, transform=ax.transAxes, fontsize=8, color=INK2, va='bottom')
    if xl: ax.set_xlabel(xl)
    if yl: ax.set_ylabel(yl)


df = pd.read_csv(os.path.join(OUT, 'greenpeft_ml_ready_dataset.csv'))
cfg = pd.read_csv(os.path.join(OUT, '_config_level_analysis.csv'))
metrics = json.load(open(os.path.join(OUT, 'greenpeft_cv_metrics.json')))
bundle = joblib.load(os.path.join(OUT, 'greenpeft_surrogate_models.joblib'))
raw = df[df.analysis_role == 'surrogate_train'].copy()
ok = cfg[cfg.fits == 1].copy()
saved = []


def save(fig, name):
    p = os.path.join(FIG, name)
    fig.savefig(p); plt.close(fig); saved.append(name)
    print(f'  {name}')


# ===== FIG 1: provenance & missingness =====
cols = ['method', 'rank', 'is_adapter_method', 'trainable_param_pct', 'gpu_total_gb',
        'accuracy', 'peak_gpu_memory_gb', 'energy_kwh', 'wall_clock_seconds']
M = pd.read_csv(os.path.join(OUT, '_missingness_by_source.csv'), index_col=0).loc[cols]
fig, ax = plt.subplots(figsize=(7.2, 4.2))
im = ax.imshow(M.values, cmap=SEQ, vmin=0, vmax=100, aspect='auto')
ax.set_xticks(range(M.shape[1]))
ax.set_xticklabels([c.replace('_', '\n') for c in M.columns], fontsize=8)
ax.set_yticks(range(len(M))); ax.set_yticklabels(M.index, fontsize=8)
for i in range(M.shape[0]):
    for j in range(M.shape[1]):
        v = M.values[i, j]
        ax.text(j, i, f'{v:.0f}', ha='center', va='center', fontsize=8,
                color='white' if v > 55 else INK)
ax.grid(False)
fig.colorbar(im, ax=ax, label='% missing', fraction=0.03, pad=0.02)
finish(ax, 'Missingness is block-structured by source, not random',
       'Each source measures a different thing; 100% blocks are MNAR-by-design, so cross-block imputation would fabricate data')
save(fig, 'fig01_missingness_by_source.png')

# ===== FIG 2: measurement-regime audit =====
p = raw[raw.status == 'ok'].copy()
p['watts'] = p.energy_kwh * 3.6e6 / p.wall_clock_seconds
fig, axes = plt.subplots(1, 2, figsize=(9.6, 3.9))
ax = axes[0]
for i, (nm, lab) in enumerate([('original', 'original pass'), ('remeasured', 'remeasured pass')]):
    v = p[p.measurement_pass == nm]['watts']
    ax.scatter(np.random.default_rng(i).normal(i, 0.055, len(v)), v, s=26,
               color=SERIES[i], alpha=0.85, edgecolor='white', linewidth=0.6,
               marker=['o', 's'][i], label=f'{lab} (n={len(v)})', zorder=3)
    ax.hlines(v.mean(), i - 0.22, i + 0.22, color=INK, lw=2, zorder=4)
    ax.text(i, v.mean() + 3.5, f'{v.mean():.1f} W', ha='center', fontsize=8,
            fontweight='bold', color=INK)
ax.axhline(70, color=INK2, ls='--', lw=1)
ax.text(1.46, 71, 'T4 TDP 70 W', fontsize=7.5, color=INK2, ha='right')
ax.axhline(10, color=INK2, ls=':', lw=1)
ax.text(-0.42, 12.0, 'T4 idle ~10 W', fontsize=7.5, color=INK2, ha='left')
ax.set_xticks([0, 1]); ax.set_xticklabels(['original', 'remeasured'])
ax.set_xlim(-0.45, 1.5); ax.set_ylim(0, 85); ax.legend(fontsize=7.5, loc='center right')
finish(ax, 'Implied GPU power draw', 'Same 41 configs, two passes', None, 'watts')

ax = axes[1]
pv = p.pivot_table(index='config_id', columns='measurement_pass', values='energy_kwh',
                   aggfunc='first').dropna()
ax.scatter(pv['original'], pv['remeasured'], s=30, color=SERIES[4], alpha=0.85,
           edgecolor='white', linewidth=0.6, zorder=3)
lim = [0, pv['original'].max() * 1.08]
ax.plot(lim, lim, color=INK2, ls='--', lw=1, label='1:1 agreement')
ax.plot(lim, [x * 0.169 for x in lim], color=SERIES[1], lw=2, label='observed 0.169x')
ax.set_xlim(lim); ax.set_ylim(0, pv['remeasured'].max() * 1.15)
ax.legend(fontsize=7.5, loc='upper left')
finish(ax, 'Paired energy: the two passes disagree 6x',
       f'Pearson r = {pv["original"].corr(pv["remeasured"]):.3f} (ranking preserved), but scale is not',
       'energy_kwh, original pass', 'energy_kwh, remeasured')
fig.suptitle('Measurement-regime audit: the "remeasured" pass recorded the idle floor, not the training load',
             x=0.5, y=1.06, fontsize=10.5, fontweight='bold', color=INK)
save(fig, 'fig02_measurement_regime_audit.png')

# ===== FIG 3: Pareto front, accuracy vs energy =====
fig, ax = plt.subplots(figsize=(6.6, 4.4))
e = ok[ok.energy_kwh.notna()]
for m in METHODS:
    s = e[e.method == m]
    if not len(s): continue
    ax.scatter(s.energy_kwh * 1000, s.accuracy, s=62, color=MC[m], marker=MARK[m],
               alpha=0.9, edgecolor='white', linewidth=0.8, label=m, zorder=3)
pts = e[['energy_kwh', 'accuracy']].values
front = [i for i in range(len(pts)) if not np.any(
    (pts[:, 0] <= pts[i, 0]) & (pts[:, 1] >= pts[i, 1]) &
    ((pts[:, 0] < pts[i, 0]) | (pts[:, 1] > pts[i, 1])))]
fp = e.iloc[front].sort_values('energy_kwh')
ax.plot(fp.energy_kwh * 1000, fp.accuracy, color=INK2, lw=1.4, ls='--',
        zorder=2, label='Pareto front')
ax.legend(fontsize=8, ncol=2, loc='lower right')
finish(ax, 'Accuracy vs training energy', f'{len(e)} feasible configs; {len(fp)} are Pareto-optimal',
       'training energy (Wh)', 'SST-2 accuracy')
save(fig, 'fig03_pareto_accuracy_energy.png')

# ===== FIG 4: per-method profile across tiers =====
panels = [('accuracy', 'accuracy', 1), ('peak_gpu_memory_gb', 'peak VRAM (GB)', 1),
          ('energy_kwh', 'energy (Wh)', 1000), ('wall_clock_seconds', 'wall-clock (s)', 1)]
fig, axes = plt.subplots(2, 2, figsize=(10, 6.6))
for ax, (col, lab, sc) in zip(axes.ravel(), panels):
    w, mm = 0.15, [m for m in METHODS if m in ok.method.unique()]
    for k, m in enumerate(mm):
        xs, ys = [], []
        for ti, t in enumerate(TIERS):
            s = ok[(ok.method == m) & (ok.backbone == t)][col].dropna()
            if len(s):
                xs.append(ti + (k - len(mm) / 2 + 0.5) * w); ys.append(s.mean() * sc)
        if not xs:
            continue
        if col == 'accuracy':
            # Accuracy spans 0.89-0.95. A bar implies a zero baseline; truncating one to
            # show that range is the classic misleading-axis anti-pattern. A dot carries
            # no baseline claim, so the narrow range can be shown honestly.
            ax.scatter(xs, ys, s=58, color=MC[m], marker=MARK[m], label=m, zorder=3,
                       edgecolor='white', linewidth=0.8)
        else:
            ax.bar(xs, ys, width=w * 0.86, color=MC[m], label=m, zorder=3,
                   edgecolor='white', linewidth=1.0)
    ax.set_xticks(range(len(TIERS))); ax.set_xticklabels(TIERS)
    ax.set_ylabel(lab)
    if col == 'accuracy':
        ax.set_ylim(0.87, 0.96)
        for ti in range(len(TIERS) - 1):
            ax.axvline(ti + 0.5, color='#e9e9e4', lw=0.8, zorder=1)
    finish(ax, lab, None)
axes[0, 0].legend(fontsize=7.5, ncol=5, loc='upper left', bbox_to_anchor=(0, 1.32))
fig.suptitle('Method profile across model-scale tiers (feasible configs only)',
             x=0.5, y=1.03, fontsize=11, fontweight='bold', color=INK)
fig.tight_layout()
save(fig, 'fig04_method_profiles.png')

# ===== FIG 5: surrogate parity plots =====
fig, axes = plt.subplots(1, 4, figsize=(14, 3.6))
for ax, tgt in zip(axes, ['accuracy', 'peak_gpu_memory_gb', 'energy_kwh', 'wall_clock_seconds']):
    d = pd.read_csv(os.path.join(OUT, f'_cvpred_{tgt}.csv'))
    mt = metrics['regression'][tgt]
    for i, t in enumerate(TIERS):
        s = d[d.backbone == t]
        if len(s):
            ax.scatter(s.y_true, s.y_pred, s=44, color=SERIES[i], marker=MARK[METHODS[i]],
                       alpha=0.88, edgecolor='white', linewidth=0.7, label=t, zorder=3)
    lo = min(d.y_true.min(), d.y_pred.min()); hi = max(d.y_true.max(), d.y_pred.max())
    pad = (hi - lo) * 0.08
    ax.plot([lo - pad, hi + pad], [lo - pad, hi + pad], color=INK2, ls='--', lw=1, zorder=2)
    ax.set_xlim(lo - pad, hi + pad); ax.set_ylim(lo - pad, hi + pad)
    finish(ax, tgt, f'pooled R²={mt["pooled_r2"]:+.2f} · MAPE={mt["mape_pct"]:.1f}%',
           'measured', 'predicted')
axes[0].legend(fontsize=7.5, title='held-out tier', title_fontsize=7.5, loc='upper left')
fig.suptitle('Surrogate accuracy under leave-one-tier-out CV (each point predicted by a model that never saw its scale tier)',
             x=0.5, y=1.08, fontsize=11, fontweight='bold', color=INK)
fig.tight_layout()
save(fig, 'fig05_surrogate_parity_loto.png')

# ===== FIG 6: permutation importance =====
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, tgt in zip(axes, ['peak_gpu_memory_gb', 'energy_kwh']):
    sub = ok[ok[tgt].notna()]
    r = permutation_importance(bundle['regressors'][tgt], sub[bundle['features']['all']],
                               sub[tgt], n_repeats=30, random_state=42, n_jobs=-1)
    imp = pd.Series(r.importances_mean, index=bundle['features']['all']).nlargest(10)[::-1]
    ax.barh(range(len(imp)), imp.values, color=SERIES[0], height=0.68, zorder=3)
    ax.set_yticks(range(len(imp))); ax.set_yticklabels(imp.index, fontsize=8)
    for i, v in enumerate(imp.values):
        ax.text(v + imp.max() * 0.015, i, f'{v:.3f}', va='center', fontsize=7.5, color=INK2)
    ax.set_xlim(0, imp.max() * 1.18)
    finish(ax, tgt, 'permutation importance (top 10)', 'mean R² drop when shuffled')
fig.suptitle('What the surrogate actually uses', x=0.5, y=1.03, fontsize=11,
             fontweight='bold', color=INK)
fig.tight_layout()
save(fig, 'fig06_feature_importance.png')

# ===== FIG 7: carbon and grid-intensity sensitivity =====
# carbon = energy x grid_intensity exactly, so a carbon-vs-method chart is the energy chart
# with a rescaled axis. What carbon adds that energy does not is the GRID term: the same run
# emits very different amounts depending on where it is executed.
GRIDS = [('Iceland', 0.028), ('Sweden', 0.045), ('France', 0.056), ('EU avg', 0.251),
         ('Global avg', 0.475), ('This study', 0.650), ('India', 0.713), ('Coal-heavy', 0.820)]
fig, axes = plt.subplots(1, 2, figsize=(11.4, 4.3))

ax = axes[0]
mm = [m for m in METHODS if m in ok.method.unique()]
cb = [(m, ok[ok.method == m].energy_kwh.mean() * GRID_KG * 1000) for m in mm]
cb.sort(key=lambda r: r[1])
ax.barh(range(len(cb)), [v for _, v in cb], color=[MC[m] for m, _ in cb],
        height=0.66, zorder=3, edgecolor='white', linewidth=1.0)
ax.set_yticks(range(len(cb))); ax.set_yticklabels([m for m, _ in cb])
for i, (m, v) in enumerate(cb):
    ax.text(v + 0.05, i, f'{v:.2f} g', va='center', fontsize=8.5, color=INK2)
ax.set_xlim(0, max(v for _, v in cb) * 1.22)
finish(ax, 'Carbon per fine-tuning run',
       f'grid intensity {GRID_KG} kgCO2/kWh, as measured', 'gCO2eq per run')

ax = axes[1]
xs = np.linspace(0, 0.85, 50)
for m in mm:
    e = ok[ok.method == m].energy_kwh.mean()
    ax.plot(xs, e * xs * 1000, color=MC[m], lw=2, zorder=3, label=m)
    _dy = 1.055 if m == 'lora_fa' else (0.95 if m == 'lora' else 1.0)
    ax.text(0.845, e * 0.845 * 1000 * _dy, f' {m}', fontsize=7.5, color=INK2,
            va='center')
for nm, ci in GRIDS:
    ax.axvline(ci, color='#e9e9e4', lw=0.8, zorder=1)
lo = ok[ok.method == 'lora'].energy_kwh.mean() * 0.820 * 1000
fu = ok[ok.method == 'full_ft'].energy_kwh.mean() * 0.028 * 1000
ax.scatter([0.820, 0.028], [lo, fu], s=70, color=INK, zorder=5, marker='*')
ax.annotate(f'LoRA on a coal-heavy grid: {lo:.2f} g\nis {lo/fu:.0f}x full fine-tuning\nin Iceland ({fu:.2f} g)',
            xy=(0.820, lo), xytext=(0.115, lo * 1.32), fontsize=7.5, color=INK,
            arrowprops=dict(arrowstyle='->', color=INK2, lw=0.9))
TICKS = [('Iceland', 0.028), ('EU avg', 0.251), ('Global avg', 0.475),
         ('This study', 0.650), ('Coal-heavy', 0.820)]
ax.set_xticks([ci for _, ci in TICKS])
ax.set_xticklabels([f'{nm}\n{ci:g}' for nm, ci in TICKS], fontsize=7.2)
ax.set_xlim(0, 0.93)
finish(ax, 'Where the run executes dominates which method is used',
       'carbon scales linearly with grid intensity, so the region multiplies every method equally',
       'grid carbon intensity (kgCO2/kWh)', 'gCO2eq per run')
fig.tight_layout()
save(fig, 'fig07_carbon_grid_sensitivity.png')

print(f'\n{len(saved)} figures written to {FIG}')

  fig01_missingness_by_source.png


  fig02_measurement_regime_audit.png


  fig03_pareto_accuracy_energy.png


  fig04_method_profiles.png


  fig05_surrogate_parity_loto.png


  fig06_feature_importance.png


  fig07_carbon_grid_sensitivity.png

7 figures written to C:\Users\Tanzil\Downloads\green\new update works\figures


## Outputs

| File | What it is |
|---|---|
| `greenpeft_ml_ready_dataset.csv` | **the one dataset** — 515 rows × 45 cols, role-tagged |
| `greenpeft_surrogate_models.joblib` | feasibility classifier + 4 regressors, preprocessing baked in |
| `greenpeft_cv_metrics.json` | honest LOTO metrics, ready to cite |
| `figures/fig01..fig06*.png` | publication figures at 300 dpi |

The `.joblib` bundle carries its own `ColumnTransformer`, so it consumes a raw config
`DataFrame` directly — no preprocessing to reimplement at the call site.

```python
import joblib, pandas as pd
b = joblib.load('greenpeft_surrogate_models.joblib')
b['regressors']['energy_kwh'].predict(config_df)      # kWh
b['regressors']['peak_gpu_memory_gb'].predict(config_df)
predicted_kwh * b['carbon_intensity_kg_per_kwh']      # kgCO2eq — derived, not modelled
```

### What would most improve these numbers

More **model scales**, not a fancier regressor. Four tiers (0.5/1.1/1.5/3.0B) is the binding
constraint on every extrapolation result above; the already-built 13-model zoo would roughly
triple the scale diversity. Re-running the sweep with verified energy telemetry would also
remove the need for Stage 3's regime exclusion, doubling the energy sample.